# JRC sea-level (Total Water Level) forecasts

The `earthlens.jrc` backend serves the JRC / Copernicus-EMS probabilistic,
data-driven **sea-level forecasts** — storm surge + tide + wave-derived coastal
**Total Water Level (TWL)** — alongside the European Flood Hazard Map (EFHM).
There are two products, both **global 0.25°** NetCDF-4 cubes:

- **medium-term** — issued twice daily, 15-day horizon;
- **subseasonal** — issued weekly, ~46-day horizon, with a global per-country
  **coastal-summary CSV** alongside the gridded cube.

This notebook fetches the latest medium-term forecast for the North Sea, maps
it, and reads the global coastal summary. It downloads live from the JRC
open-data server (no credentials — CC-BY-4.0); only the small area of interest
is read over `/vsicurl`, not the whole multi-gigabyte cube.

In [ ]:
import tempfile
from pathlib import Path

from pyramids.dataset import Dataset

from earthlens.core import EarthLens

OUT = Path(tempfile.mkdtemp(prefix="jrc_twl_"))
OUT

## 1. Gridded TWL forecast for a coastal area

Select `product="medium_term"` and a bounding box. `reference_time` defaults to
`"latest"`, which resolves the newest **complete** forecast cycle (the backend
walks the dated directory tree and honours the `endFls` sentinel). The default
field is `TWL75` — the 75th-percentile total water level — and every forecast
time step becomes a band of the written GeoTIFF.

In [ ]:
paths = EarthLens(
    data_source="jrc:sea-level-forecast",
    product="medium_term",
    lat_lim=[50.0, 58.0],  # North Sea
    lon_lim=[-2.0, 10.0],
    path=OUT,
).download()
paths

## 2. Map the first forecast step

Read the cropped GeoTIFF back with pyramids. It is georeferenced (EPSG:4326) and
carries one band per forecast day; land cells are `NaN`. We plot the first
forecast step.

In [ ]:
grid = Dataset.read_file(paths[0])
print("bands (forecast steps):", grid.band_count, "| epsg:", grid.epsg)
print("geotransform:", grid.geotransform)

# plot() derives the extent from the raster, so the hand-built one is gone.
glyph = grid.plot(
    band=0,
    cmap="viridis",
    title="JRC medium-term TWL75 forecast — first step (North Sea)",
)
glyph.cbar.set_label("total water level (m)")

## 3. Global coastal summary

The subseasonal product also publishes a global, per-country coastal summary as
a table. The `jrc:coastal-forecast` key returns it directly as a `pandas.DataFrame`
(exceedance probabilities against return-period thresholds, plus a 1–10 severity
summary per country).

In [ ]:
summary = EarthLens(data_source="jrc:coastal-forecast").download()
print("countries:", len(summary))
summary[["GID_0", "NAME_0", "summary_TWL_1_10"]].head(10)

## Notes

- **Distinct from the EFHM.** The `efhm` / `jrc-flood` keys serve the static
  European river-flood **depth** map (per return period); the sea-level keys here
  serve the coastal **TWL forecast**. Both live in the one `earthlens.jrc`
  backend.
- **Windowed reads.** Only the AOI window is transferred over `/vsicurl`, so a
  small area costs little of the 13–38 GB cube. A forecast cycle is chosen by
  `reference_time`, so `aggregate=` is rejected.
- **Licence.** CC-BY-4.0 (Copernicus EMS / EC JRC).